In [2]:
import requests
import pandas as pd
import time
import re
from bs4 import BeautifulSoup
from urllib.parse import quote

In [3]:
# all unique items across both baskets — scraped once
ITEMS = {
    'ovesne_vlocky':    ('ovesné vločky',    ['vločky'],           ['müsli', 'granola', 'tyčinka', 'instant'],                                                                         'g'),
    'mleko':            ('mléko polotučné',  ['mléko'],            ['kokosov', 'mandlov', 'sójov', 'ovesn', 'rýžov', 'kondenzov', 'smetana', 'kakao'],                               'ml'),
    'banany':           ('banán',            ['banán'],            ['sušen', 'chipsy', 'smoothie', 'mražen', 'mléčn', 'nápoj', 'příchuť', 'svačinka', 'ovocn', 'müsli', 'čokolád', 'koblih', 'pyré', 'jabl', 'červen', 'hrušk'], 'g'),
    'chleb':            ('chléb',            ['chléb'],            ['toast', 'toustov', 'rohlík', 'bageta', 'máslov'],                                                                  'g'),
    'maslo':            ('máslo',            ['máslo'],            ['arašídov', 'ořechov', 'pomazánka', 'margarin', 'croissant', 'chléb', 'fazole', 'sušenk', 'příchuť', 'držák', 'miska', 'koš'], 'g'),
    'vejce':            ('čerstvá vejce',    ['vejce'],            ['křepelčí', 'salát', 'vaječn', 'polévka', 'aspik', 'bílek', 'bageta', 'držák', 'miska'],                           'piece'),
    'spagety':          ('špagety',          ['špaget'],           ['instant', 'polévka', 'stroj'],                                                                                     'g'),
    'pesto':            ('pesto bazalkové',  ['pesto'],            ['těstoviny', 'chipsy', 'rajčat'],                                                                                   'g'),
    'ryze':             ('rýže dlouhozrnná', ['rýže'],             ['instant', 'cereálie', 'nápoj', 'rýžov'],                                                                          'g'),
    'mrazena_zelenina': ('mražená zelenina', ['zelen'],            ['ovoce', 'hranolky', 'knedlík', 'polévka'],                                                                        'g'),
    'kureci_prsa':      ('kuřecí prs',       ['kuřecí', 'prs'],   ['smažen', 'marinovan', 'paštika', 'šunka', 'nugety', 'konzerva', 'kočk', 'sous-vide', 'řízek', 's kostí', 'nudličky'], 'g'),
    'tofu':             ('tofu natural',     ['tofu'],             ['marinovan', 'dezert', 'tyčinka', 'uzen'],                                                                          'g'),
    'cibule':           ('cibule',           ['cibule'],           ['jarní', 'sušen', 'granulovan', 'nakládan', 'chutney'],                                                             'g'),
    'cesnek':           ('česnek',           ['česnek'],           ['sušen', 'granulovan', 'pasta', 'olej', 'prášek', 'nakládan', 'černý', 'bageta', 'pečen', 'sekan', 'hovězí', 'hotov', 'špenát', 'smetana', 'paprika', 'grill', 'pomazánka', 'nálev', 'chilli'], 'g'),
    'syr_eidam':        ('eidam',            ['eidam'],            ['taven', 'pomazánka'],                                                                                              'g'),
    'tunak':            ('tuňák přírodní',   ['tuňák'],            ['pomazánka', 'salát', 'olivov'],                                                                                   'g'),
    'jogurt':           ('bílý jogurt',      ['jogurt'],           ['pitný', 'skyr', 'sójov', 'ochucen', 'kokosov', 'mango', 'müsli'],                                                 'g'),
    'kava':             ('mletá káva 250g',  ['káva'],             ['kapsle', 'instant', 'ledov', 'zrnkov', 'bezkofein'],                                                              'g'),
    # vegan-only items
    'ovesny_napoj':     ('ovesný nápoj',     ['ovesn'],            ['čokolád', 'káva', 'vanilka', 'vločky'],                                                                           'ml'),
    'sojovy_napoj':     ('sójový nápoj',     ['sójov'],            ['čokolád', 'káva', 'vanilka', 'jahoda', 'borůvka', 'omáčka', 'rýžov'],                                            'ml'),
    'repkovy_olej':     ('řepkový olej',     ['olej'],             ['olivov', 'slunečnicov', 'kokosov', 'palmov', 'motorov', 'espres', 'česnek'],                                      'ml'),
    'cervena_cocka':    ('červená čočka',    ['čočka'],            ['polévka', 'hotov', 'konzerva'],                                                                                   'g'),
    'rajcatova_omacka': ('rajčatová omáčka', ['rajčat'],           ['polévka', 'kečup', 'lečo', 'těsto', 'cherry', 'keříkov', 'čerstvá'],                                             'g'),
    'paprika':          ('paprika',          ['paprika'],          ['mlet', 'sušen', 'kořen', 'chilli', 'omáčka', 'smažen', 'nálev', 'mražen', 'špičat', 'řezan', 'řezy', 'konzervov', 'česnek'], 'g'),
}

# standard basket — which items and how many per week
BASKET_STANDARD = {
    'ovesne_vlocky', 'mleko', 'banany', 'chleb', 'maslo', 'vejce',
    'spagety', 'pesto', 'ryze', 'mrazena_zelenina', 'kureci_prsa',
    'tofu', 'cibule', 'cesnek', 'syr_eidam', 'tunak', 'jogurt', 'kava',
}

# vegan basket — which items
BASKET_VEGAN = {
    'ovesne_vlocky', 'banany', 'chleb', 'spagety', 'ryze',
    'mrazena_zelenina', 'tofu', 'cibule', 'cesnek', 'kava',
    'ovesny_napoj', 'sojovy_napoj', 'repkovy_olej',
    'cervena_cocka', 'rajcatova_omacka', 'paprika',
}
_ = None

In [4]:
def passes_filter(name, require, exclude):
    n = name.lower()
    return all(k.lower() in n for k in require) and not any(k.lower() in n for k in exclude)


def parse_qty(text):
    if not text:
        return None, None
    t = str(text).lower()
    if m := re.search(r'(\d+)\s*ks', t):                  return int(m.group(1)), 'piece'
    if m := re.search(r'(\d+[,.]?\d*)\s*kg', t):          return float(m.group(1).replace(',','.')) * 1000, 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*g(?!\w)', t):    return float(m.group(1).replace(',','.')), 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*l(?!\w)', t):    return float(m.group(1).replace(',','.')) * 1000, 'ml'
    if m := re.search(r'(\d+)\s*ml', t):                  return int(m.group(1)), 'ml'
    return None, None


def select_best(candidates, spec):
    search, require, exclude, unit = spec
    filtered = [c for c in candidates if passes_filter(c['title'], require, exclude)]
    if not filtered:
        return None, []
    scored = []
    for c in filtered:
        qty, u = parse_qty(c.get('packaging', ''))
        ppu = c['price'] / qty if (qty and u == unit) else None
        scored.append({**c, 'price_per_unit': ppu})
    with_ppu = [s for s in scored if s['price_per_unit']]
    return min(with_ppu, key=lambda x: x['price_per_unit']) if with_ppu else min(scored, key=lambda x: x['price']), scored


def scrape(search_fn, basket):
    selected, raw_all = [], []
    for key, spec in basket.items():
        try:
            raw = search_fn(spec[0])
        except Exception as e:
            print(f'{key}... FAILED ({e})')
            selected.append({'item': key, 'title': None, 'price_czk': None, 'packaging': None, 'price_per_unit': None})
            continue
        time.sleep(1.5)
        for c in raw:
            raw_all.append({'item': key, **c})
        best, _ = select_best(raw, spec)
        if best:
            print(f'{key}... {best["title"]} | {best["price"]} Kč | {best["packaging"]}')
            selected.append({'item': key, 'title': best['title'], 'price_czk': best['price'],
                             'packaging': best['packaging'], 'price_per_unit': best['price_per_unit']})
        else:
            print(f'{key}... NO MATCH')
            selected.append({'item': key, 'title': None, 'price_czk': None, 'packaging': None, 'price_per_unit': None})
    return pd.DataFrame(selected), pd.DataFrame(raw_all)

## Rohlik

In [6]:
# Rohlik
rohlik_headers = {'User-Agent': 'JEM207 StudentProject (contact: your_email@fsv.cuni.cz)'}

def rohlik_search(term):
    url = (f'https://www.rohlik.cz/services/frontend-service/search-metadata'
           f'?search={quote(term)}&offset=0&limit=25&companyId=1'
           f'&filterData=%7B%22filters%22%3A%5B%5D%7D&canCorrect=true')
    data = requests.get(url, headers=rohlik_headers, timeout=15).json()
    return [{'title': p['productName'], 'price': p['price']['full'], 'packaging': p.get('textualAmount', '')}
            for p in data.get('data', {}).get('productList', []) if 'productName' in p]

df_rohlik, df_rohlik_raw = scrape(rohlik_search, ITEMS)

ovesne_vlocky... Yutto Celozrnné ovesné vločky extra jemné | 11.9 Kč | 500 g
mleko... Miil Trvanlivé polotučné mléko 1,5% | 14.9 Kč | 1 l
banany... Banán 1 ks | 6.95 Kč | cca 190 g
chleb... Merhautovo pekařství Chléb žitný | 35.9 Kč | 400 g
maslo... Miil Máslo 82% | 44.9 Kč | 250 g
vejce... Pohodová vejce Podestýlková L | 45.9 Kč | 6 ks
spagety... Špagety | 12.9 Kč | 400 g
pesto... Kitchin Bazalkové pesto alla Genovese | 37.9 Kč | 190 g
ryze... Kitchin Rýže dlouhozrnná | 29.9 Kč | 1 kg
mrazena_zelenina... Agro Jesenice Zelenina s kukuřicí | 20.9 Kč | 350 g
kureci_prsa... Kuřecí prsní řízky od českého výrobce XXL balení | 263.88 Kč | cca 1,2 kg
tofu... SoYum Tofu | 60.9 Kč | 450 g
cibule... Cibule žlutá, síť | 19.9 Kč | 1 kg
cesnek... Česnek "ošklivák" český, síť | 69.9 Kč | 350 g
syr_eidam... Miil Eidam 30% bloček | 44.9 Kč | 250 g
tunak... Kitchin Tuňák celý ve slunečnicovém oleji | 52.9 Kč | 195 g
jogurt... Hollandia Nízkotučný jogurt bílý | 17.9 Kč | 380 g
kava... Ubomi Intense RFA 

## Košík

In [7]:
# Kosik
def kosik_search(term):
    url = (f'https://www.kosik.cz/api/front/page/products/flexible'
           f'?vendor=1&slug=vyhledavani&limit=30&search_term={quote(term)}&platform=web')
    data = requests.get(url, headers=rohlik_headers, timeout=15).json()
    out = []
    for p in data.get('products', {}).get('items', []):
        q = p.get('productQuantity') or {}
        out.append({'title': p['name'], 'price': p['price'],
                    'packaging': f"{q.get('value','')} {q.get('unit','')}".strip()})
    return out

df_kosik, df_kosik_raw = scrape(kosik_search, ITEMS)

ovesne_vlocky... Vločkárna Vřesce Ovesné vločky | 21.9 Kč | 450.0 g
mleko... Fine Life Polotučné trvanlivé mléko 1,5% | 9.9 Kč | 1.0 l
banany... Banán, 1 ks | 6.783 Kč | 170.0 g
chleb... Český pekař Chléb konzumní | 42.5 Kč | 1.2 kg
maslo... Tatra Farmářské máslo (84%) | 34.9 Kč | 200.0 g
vejce... Vejce Kosičky Čerstvá podestýlková vejce, vel. M | 79.9 Kč | 10.0 ks
spagety... ARO Špagety bezvaječné | 12.9 Kč | 500.0 g
pesto... Casino Bazalkové pesto | 64.9 Kč | 190.0 g
ryze... ARO Rýže dlouhozrnná parboiled | 34.9 Kč | 1.0 kg
mrazena_zelenina... ARO  Zeleninová směs pod svíčkovou | 19.9 Kč | 400.0 g
kureci_prsa... Fine Life Kuřecí prsní řízky | 94.9 Kč | 500.0 g
tofu... Lunter Tofu natural | 23.9 Kč | 180.0 g
cibule... Cibule žlutá, síť | 19.9 Kč | 1.0 kg
cesnek... Česnek drcený porce | 64.9 Kč | 1.0 kg
syr_eidam... Fine Life Eidam 30 % bloček | 35.9 Kč | 200.0 g
tunak... Giana Tuňák drcený ve vlastní šťávě | 37.9 Kč | 170.0 g
jogurt... ARO Jogurt bílý 1,5% tuku | 19.9 Kč | 400.0 g
kav

## Billa

In [8]:
# Billa
billa_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def billa_search(term):
    url = f'https://www.billa.cz/vyhledavani/{quote(term)}?tab=products'
    soup = BeautifulSoup(requests.get(url, headers=billa_headers, timeout=15).text, 'html.parser')
    out, seen = [], set()
    for card in soup.find_all('a', href=lambda h: h and '/produkt/' in h):
        try:
            title = card.get_text(' ', strip=True)
            if not title or title in seen:
                continue
            parent = card.find_parent('li')
            if not parent:
                continue
            chunks = [c.replace('\xa0',' ').strip() for c in parent.get_text(separator='|',strip=True).split('|') if c.strip()]
            packaging = next((c for c in chunks if re.match(r'^\d+[,.]?\d*\s*(ks|kg|g|ml|l)\b$', c, re.I)), '')
            price = None
            for chunk in chunks:
                if re.match(r'^\d+[,.]?\d*\s*(ks|kg|g|ml|l)\b', chunk, re.I): continue
                if 'klub' in chunk.lower(): continue
                m = re.search(r'(\d+(?:[,.]\d+)?)\s*Kč', chunk)
                if m:
                    price = float(m.group(1).replace(',','.'))
                    break
            if price:
                seen.add(title)
                out.append({'title': title, 'price': price, 'packaging': packaging})
        except Exception:
            continue
    return out

df_billa, df_billa_raw = scrape(billa_search, ITEMS)

ovesne_vlocky... clever Ovesné vločky celozrnné 500g | 13.9 Kč | 500 g
mleko... clever Mléko polotučné trvanlivé 1,5 % tuku 1l | 12.9 Kč | 1 l
banany... CORNY proteinová tyčinka 30% banán 50g | 51.9 Kč | 50 g
chleb... Chléb Chalupářský, 850g kulatý | 58.9 Kč | 850 g
maslo... Milkpol Máslo 82% 250 g | 39.9 Kč | 250 g
vejce... BILLA Čerstvá vejce L 6 ks | 34.9 Kč | 6 ks
spagety... Franz Josef Kaiser Exclusive Špagety 500g | 29.9 Kč | 500 g
pesto... clever Bazalkové pesto 190g | 29.9 Kč | 190 g
ryze... BILLA Dlouhozrnná rýže loupaná 1000g | 44.9 Kč | 1 kg
mrazena_zelenina... clever Zeleninová směs podsvíčková 450g | 19.9 Kč | 450 g
kureci_prsa... VOCÍLKA Kuřecí prsní řízky premium | 259.9 Kč | 1 kg
tofu... LUNTER Naturální tofu 180g | 22.9 Kč | 180 g
cibule... Česká Farma Cibule 1kg, síť | 20.9 Kč | 1 kg
cesnek... Česká Farma Česnek síťka, 200g | 49.9 Kč | 200 g
syr_eidam... clever Eidam bloček 30% 250g | 44.9 Kč | 250 g
tunak... Tuňák přírodní Vier Diamanten 195 g | 77.82 Kč | 195 g
jogu

## Lidl

In [9]:
# Lidl
def lidl_search(term):
    url = (f'https://www.lidl.cz/q/api/search'
           f'?q={quote(term)}&locale=cs_CZ&assortment=CZ&version=2.1.0&fetchsize=20')
    data = requests.get(url, headers=rohlik_headers, timeout=15).json()
    out = []
    for item in data.get('items', []):
        try:
            d = item['gridbox']['data']
            out.append({'title': d.get('fullTitle', d.get('title','')),
                        'price': d['price']['price'],
                        'packaging': d['price'].get('packaging',{}).get('text','')})
        except (KeyError, TypeError):
            continue
    return out

df_lidl, df_lidl_raw = scrape(lidl_search, ITEMS)

ovesne_vlocky... Ovesné vločky hrub | 11.9 Kč | 500 g
mleko... Trvanlivé mléko 1,5% | 12.9 Kč | 1 l
banany... NO MATCH
chleb... Dřevorubecký chléb | 23.9 Kč | 405 g
maslo... Perla Margarín s máslovou příchutí 39% tuku | 34.9 Kč | 450 g
vejce... NO MATCH
spagety... Špagety | 11.9 Kč | 500 g
pesto... NO MATCH
ryze... FAILED (Expecting value: line 1 column 1 (char 0))
mrazena_zelenina... NO MATCH
kureci_prsa... BIO Kuřecí prsní řízky | 59.9 Kč | Dostupné pouze ve vybraných prodejnách
tofu... NO MATCH
cibule... NO MATCH
cesnek... Česnek | 39.9 Kč | 
syr_eidam... PILOS Eidam | 9.9 Kč | 
tunak... Tuňák žlutoploutvý | 39.9 Kč | cena za 100 g
jogurt... Bílý jogurt | 17.9 Kč | 300 g
kava... NO MATCH
ovesny_napoj... NO MATCH
sojovy_napoj... NO MATCH
repkovy_olej... Řepkový olej | 34.9 Kč | 1 l
cervena_cocka... NO MATCH
rajcatova_omacka... NO MATCH
paprika... NO MATCH


In [13]:
import os
os.makedirs('../data/raw', exist_ok=True)

df_rohlik.to_csv('../data/raw/rohlik_prices.csv', index=False)
df_kosik.to_csv('../data/raw/kosik_prices.csv', index=False)
df_billa.to_csv('../data/raw/billa_prices.csv', index=False)
df_lidl.to_csv('../data/raw/lidl_prices.csv', index=False)

df_rohlik_raw.to_csv('../data/raw/rohlik_raw.csv', index=False)
df_kosik_raw.to_csv('../data/raw/kosik_raw.csv', index=False)
df_billa_raw.to_csv('../data/raw/billa_raw.csv', index=False)
df_lidl_raw.to_csv('../data/raw/lidl_raw.csv', index=False)